# Module 2: Data Cleaning & Transformation

## Project: Hospital Operations & Patient Analytics Dashboard

### Objectives
- Load the raw dataset generated in Module 1
- Explore the dataset structure
- Check duplicates and missing values
- Standardize data
- Convert data types
- Generate Tableau-ready dataset

In [68]:
# ============================================================
# Import Required Libraries
# ============================================================

# Import pandas for data manipulation
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

In [69]:
# ============================================================
# Load Raw Dataset
# ============================================================

# Load the raw dataset generated in Module 1
df = pd.read_csv("../data/hospital_raw_data.csv")

# Display first five records
df.head()

,Patient ID,Age,Gender,Department,Diagnosis,Severity_Level,Admission Date,Discharge Date,Length of Stay,Wait_Time_Minutes,...,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%,Bed_Occupancy_Rate_Calc,ICU_Occupancy_Rate_Calc,Staff_Utilization_Calc,Readmission_Flag,Transferred_Flag,Equipment_InUse_Flag
0,PAT000001,23,Male,Gynecology,Cancer,Low,2024-01-01,2024-01-08,7,79,...,598,0.0,43.0,0.2,0.2,80.6,43.0,0,0,0
1,PAT000002,48,Male,Neurology,Kidney Disease,Medium,2024-01-01,2024-01-05,4,167,...,583,0.2,49.4,0.2,0.2,41.0,49.4,0,0,0
2,PAT000003,63,Male,Neurology,Stroke,Low,2024-01-01,2024-01-14,13,281,...,590,0.0,30.7,0.2,0.2,23.7,30.7,0,0,1
3,PAT000004,85,Male,Emergency,Asthma,Medium,2024-01-01,2024-01-12,11,271,...,597,0.2,28.5,0.4,0.4,67.7,28.5,0,1,0
4,PAT000005,93,Female,Neurology,Pneumonia,Medium,2024-01-01,2024-01-09,8,112,...,576,0.0,20.1,0.6,0.6,39.0,20.1,0,0,1


In [70]:
# ============================================================
# Initial Dataset Exploration
# ============================================================

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

# Display dataset shape
print("\nDataset Shape:")
print(df.shape)

# Display dataset information
print("\nDataset Information:")
df.info()

# Display missing values
print("\nMissing Values:")
print(df.isnull().sum())

DATASET OVERVIEW

Dataset Shape:
(100000, 58)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 58 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Patient ID                      100000 non-null  object 
 1   Age                             100000 non-null  int64  
 2   Gender                          100000 non-null  object 
 3   Department                      100000 non-null  object 
 4   Diagnosis                       100000 non-null  object 
 5   Severity_Level                  100000 non-null  object 
 6   Admission Date                  100000 non-null  object 
 7   Discharge Date                  100000 non-null  object 
 8   Length of Stay                  100000 non-null  int64  
 9   Wait_Time_Minutes               100000 non-null  int64  
 10  Doctor                          100000 non-null  object 
 11  Insurance P

## Duplicate Record Check

In [71]:
# ============================================================
# Check and Remove Duplicate Records
# ============================================================

duplicates = df.duplicated().sum()

print("Duplicate Records :", duplicates)

# Remove duplicate rows
df = df.drop_duplicates()

print("Dataset Shape After Removing Duplicates:")
print(df.shape)

Duplicate Records : 0
Dataset Shape After Removing Duplicates:
(100000, 58)


## Missing Value Analysis

In [72]:
# ============================================================
# Check Missing Values
# ============================================================

missing_values = df.isnull().sum()

missing_values = missing_values[missing_values > 0]

print(missing_values)

Transfer_Date    82400
dtype: int64


In [73]:
# ============================================================
# Handle Missing Values
# ============================================================

# Replace missing values in Transfer_Date
# Missing values indicate that the patient was not transferred.

df["Transfer_Date"] = df["Transfer_Date"].fillna("Not Transferred")

print("Missing values handled successfully.")

Missing values handled successfully.


In [74]:
# ============================================================
# Verify Missing Values
# ============================================================

print(df["Transfer_Date"].isnull().sum())

0


### Observation

The `Transfer_Date` column contained missing values because many patients were not transferred to another department. These missing values were replaced with **"Not Transferred"** to make the dataset more readable and suitable for analysis.

## Data Standardization

In [75]:
# ============================================================
# Standardize Text Columns
# ============================================================

text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].str.strip()

columns = [
    "Department",
    "Hospital Name",
    "Hospital Type",
    "City",
    "State",
    "Diagnosis",
    "Treatment",
    "Equipment",
    "Insurance Provider"
]

for col in columns:
    if col in df.columns:
        df[col] = df[col].str.title()

print("Text columns standardized successfully.")

Text columns standardized successfully.


## Data Type Conversion

In [76]:
# ============================================================
# Convert Date Columns
# ============================================================

date_columns = [
    "Admission Date",
    "Discharge Date"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Date columns converted successfully.")

Date columns converted successfully.


In [77]:
object_cols = df.select_dtypes(include="object").columns

for col in object_cols:
    df[col] = df[col].str.strip()

In [78]:
df["Equipment"] = df["Equipment"].replace({
    "Ecg Machine": "ECG Machine",
    "Mri Scanner": "MRI Scanner"
})

In [79]:
print("Equipment names standardized successfully.")

Equipment names standardized successfully.


In [80]:
df["Department"] = df["Department"].replace({
    "Icu": "ICU",
    "icu": "ICU"
})

In [81]:
print("Department names standardized successfully.")

Department names standardized successfully.


In [82]:
numeric_columns = [
    "Age",
    "Beds Available",
    "ICU Beds",
    "Billing Amount",
    "Length of Stay"
]

print(df[numeric_columns].describe())

                 Age  Beds Available  Billing Amount  Length of Stay
count  100000.000000   100000.000000   100000.000000   100000.000000
mean       47.426220    20591.271250     4893.894900        7.503140
std        27.729468    22249.351219     7710.109305        4.023773
min         0.000000      250.000000      100.000000        1.000000
25%        23.000000     2666.000000      662.000000        4.000000
50%        47.000000    13527.000000     2142.000000        7.000000
75%        71.000000    39511.000000     5002.250000       11.000000
max        95.000000    72616.000000    49999.000000       14.000000


In [83]:
print((df["Age"] < 0).sum())
print((df["Age"] > 120).sum())
print((df["Billing Amount"] < 0).sum())
print((df["Billing Amount"] >  1000000).sum())
print((df["Length of Stay"] < 0).sum())

0
0
0
0
0


In [84]:
calculated = (
    df["Discharge Date"] -
    df["Admission Date"]
).dt.days

print((calculated != df["Length of Stay"]).sum())

0


In [85]:
print(df["Gender"].value_counts())

print(df["Department"].value_counts())

print(df["Outcome"].value_counts())

Gender
Male      50406
Female    49594
Name: count, dtype: int64
Department
Dermatology        10205
Oncology           10063
Neurology          10049
Cardiology         10025
Emergency          10002
Gynecology         10001
ICU                 9987
Orthopedics         9939
General Surgery     9889
Pediatrics          9840
Name: count, dtype: int64
Outcome
Recovered      68568
Improved       20602
Transferred     8397
Deceased        2433
Name: count, dtype: int64


## Final Validation

In [86]:
# ============================================================
# Final Validation
# ============================================================

print("Dataset Shape :", df.shape)
print(df.info())
print(df.describe())
print("\nRemaining Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

Dataset Shape : (100000, 58)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 58 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   Patient ID                      100000 non-null  object        
 1   Age                             100000 non-null  int64         
 2   Gender                          100000 non-null  object        
 3   Department                      100000 non-null  object        
 4   Diagnosis                       100000 non-null  object        
 5   Severity_Level                  100000 non-null  object        
 6   Admission Date                  100000 non-null  datetime64[ns]
 7   Discharge Date                  100000 non-null  datetime64[ns]
 8   Length of Stay                  100000 non-null  int64         
 9   Wait_Time_Minutes               100000 non-null  int64         
 10  Doctor                      

## Export Cleaned Dataset

In [87]:
# ============================================================
# Export Cleaned Dataset
# ============================================================

df.to_csv("../data/hospital_cleaned.csv", index=False)

print("hospital_cleaned.csv created successfully.")

hospital_cleaned.csv created successfully.
